# MobileI2V — Génération IA sur Kaggle (GPU gratuit)
**Pas de Colab, pas de serveur, pas d'abonnement.** Dans Kaggle Notebook : Settings → Accelerator → GPU P100 ou T4 (si disponible), Internet → On. Exécuter toutes les cellules dans l'ordre. Télécharger le ZIP via Output.


In [ ]:
%pip -q install 'diffusers>=0.32,<1' 'transformers>=4.44,<5' 'accelerate>=0.33,<2' 'safetensors>=0.4,<1' 'pillow>=10,<13'
import torch
assert torch.cuda.is_available(), 'GPU Kaggle indisponible : activer GPU dans Settings ou attendre du quota.'
print('GPU :', torch.cuda.get_device_name(0))


In [ ]:
from PIL import Image
import io, re, zipfile, os
from IPython.display import display
from diffusers import AutoPipelineForImage2Image
from ipywidgets import FileUpload
import ipywidgets as widgets
upload = FileUpload(accept='image/*', multiple=False)
display(upload)
print('Sélectionner une photo ci-dessus, puis exécuter la cellule suivante.')


In [ ]:
assert upload.value, 'Importer une photo dans la cellule précédente'
item = next(iter(upload.value))
entry = upload.value[item] if isinstance(upload.value, dict) else item
raw = entry['content'] if isinstance(entry, dict) else entry.content
reference = Image.open(io.BytesIO(bytes(raw))).convert('RGB')
reference.thumbnail((512,512),Image.Resampling.LANCZOS)
w = max(64,round(reference.width/8)*8)
h = max(64,round(reference.height/8)*8)
reference = reference.resize((w,h),Image.Resampling.LANCZOS)
display(reference)


In [ ]:
# Modifier uniquement ces paramètres pour le premier essai.
histoire = 'Le personnage marche puis court puis tombe au sol'
nombre_images = 6  # Commencer petit pour économiser le quota GPU
force_reference = 0.85
assert 2 <= nombre_images <= 24
model = AutoPipelineForImage2Image.from_pretrained('stabilityai/sd-turbo', torch_dtype=torch.float16)
model.to('cuda')
model.set_progress_bar_config(disable=True)
print('Modèle chargé')


In [ ]:
def action_at(story, position):
    phases = [x.strip() for x in re.split(r'(?i)\b(?:et puis|puis|ensuite|then|afterwards|suddenly)\b|[.;\n]',story) if x.strip()]
    return phases[min(len(phases)-1,int(min(position,0.999999)*len(phases)))] if phases else story

images = []
previous = reference
for i in range(nombre_images):
    phase = i/nombre_images
    current_action = action_at(histoire,phase)
    anchor = min(0.4,max(0.1,force_reference*0.35))
    input_image = Image.blend(previous,reference,anchor)
    prompt = f'{histoire}. Current action: {current_action}. Keyframe {i+1}/{nombre_images}. Same person, same clothes, coherent anatomy, changed body pose.'
    with torch.inference_mode():
        image = model(prompt=prompt,image=input_image,strength=0.55,guidance_scale=0.0,num_inference_steps=2).images[0]
    images.append(image)
    previous = image
    print(f'Image {i+1}/{nombre_images}: {current_action}')
    display(image.resize((min(256,image.width),round(image.height*min(256,image.width)/image.width))))


In [ ]:
archive_name='/kaggle/working/mobilei2v_images_ia.zip'
with zipfile.ZipFile(archive_name,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for i,img in enumerate(images):
        b=io.BytesIO();img.save(b,format='PNG');z.writestr(f'frame_{i:04d}.png',b.getvalue())
    z.writestr('prompt.txt',histoire)
print('ZIP prêt dans Output :',archive_name)
